# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

***

Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками

In [7]:
!wget https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4 -O /kaggle/working/slow_traffic_small.mp4

--2025-05-18 15:10:38--  https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4
Resolving www.bogotobogo.com (www.bogotobogo.com)... 173.254.30.214
Connecting to www.bogotobogo.com (www.bogotobogo.com)|173.254.30.214|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2018126 (1.9M) [video/mp4]
Saving to: ‘/kaggle/working/slow_traffic_small.mp4’

/kaggle/working/slo 100%[===================>]   1.92M  2.15MB/s    in 0.9s    

2025-05-18 15:10:40 (2.15 MB/s) - ‘/kaggle/working/slow_traffic_small.mp4’ saved [2018126/2018126]



In [8]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:** 
- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV

### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**
Постоянство яркости. Интенсивность пикселя, соответствующего определенной точке объекта, остается постоянной между двумя последовательными кадрами, несмотря на его смещение. 

Это предположение является важным, так как оно позволяет связать интенсивности пикселей на разных кадрах и сформировать основное уравнение оптического потока.

Малость смещений.
Важность: Это предположение позволяет линеаризовать уравнение оптического потока с помощью разложения в ряд Тейлора. Если смещения велики, линеаризация становится неточной, и алгоритм может не сойтись к правильному решению или дать большую ошибку.


Пространственная когерентность. Пиксели в небольшой локальной окрестности вокруг рассматриваемой точки имеют схожее движение.
Важность:Предположение о локально постоянном потоке позволяет использовать информацию из окрестности точки. Мы можем составить систему уравнений для всех пикселей в окне.Без этого предположения для каждой точки было бы недостаточно информации для определения полного вектора движения.

### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:**

Пирамидальный подход в алгоритме Лукаса-Канаде нужен в первую очередь для решения проблемы больших смещений объектов между кадрами.

Пирамидальный подход эффективно расширяет диапазон смещений, с которыми может справляться алгоритм Лукаса-Канаде, делая его более применимым для реальных видео.

### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**

Если в локальном окне, используемом для вычисления потока, присутствует только однородная текстура, то невозможно однозначно определить полный вектор движения 
(dx,dy). Можно определить только компоненту движения, перпендикулярную этому краю. Движение вдоль края остается неопределенным.

Алгоритм базируется на предположении о постоянстве яркости, если это предположение не выполняется, то алгоритм будет допускать много ошибок

### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.

In [9]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

def build_image_pyramid(image, num_levels, scale_factor=0.5):
    """
    Создаёт пирамиду изображений с уменьшающимся разрешением.
    """
    pyramid = [image.copy()]
    current_image = image.copy()
    for _ in range(1, num_levels):
        new_height = int(current_image.shape[0] * scale_factor)
        new_width = int(current_image.shape[1] * scale_factor)
        if new_height < 1 or new_width < 1: # Предотвращение слишком маленьких размеров
            break
        current_image = cv2.resize(current_image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
        pyramid.append(current_image)
    return pyramid


def compute_image_gradients(image):
    """
    Вычисляет пространственные градиенты изображения.
    """
    image_float = image.astype(np.float32)
    Ix = cv2.Sobel(image_float, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image_float, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix_window, Iy_window, It_window, min_eigenvalue_threshold=1e-4):
    """
    Вычисляет оптический поток по методу Lucas-Kanade для одного окна.
    Ix_window, Iy_window, It_window - это уже вырезанные окна градиентов.
    """
    if Ix_window.size == 0 or Iy_window.size == 0 or It_window.size == 0:
        return None, None # Недостаточно данных в окне

    Ix_flat = Ix_window.flatten()
    Iy_flat = Iy_window.flatten()
    It_flat = It_window.flatten()

    A = np.zeros((2, 2), dtype=np.float64)
    A[0, 0] = np.sum(Ix_flat * Ix_flat)
    A[0, 1] = np.sum(Ix_flat * Iy_flat)
    A[1, 0] = A[0, 1]
    A[1, 1] = np.sum(Iy_flat * Iy_flat)

    b = np.zeros(2, dtype=np.float64)
    b[0] = -np.sum(Ix_flat * It_flat)
    b[1] = -np.sum(Iy_flat * It_flat)

    # Проверка обусловленности матрицы A
    eigenvalues = np.linalg.eigvals(A)
    if np.min(eigenvalues) < min_eigenvalue_threshold or np.linalg.cond(A) > 1000: # Добавил проверку числа обусловленности
        return None, None

    try:
        # Решаем систему A * v = b  => v = A_inv * b
        # v = [u, v]^T
        flow_uv = np.linalg.solve(A, b)
        u, v = flow_uv[0], flow_uv[1]
        return u, v
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    """
    Вычисляет оптический поток для патча изображения.
    window_size здесь не используется, так как предполагается, что prev_patch и curr_patch
    уже являются окнами нужного размера вокруг точки.
    """
    if prev_patch.shape != curr_patch.shape or prev_patch.size == 0:
        return None, None

    Ix, Iy = compute_image_gradients(prev_patch)
    It = curr_patch.astype(Ix.dtype) - prev_patch.astype(Ix.dtype) # Приведение типов


    u, v = compute_lk_optical_flow_point(Ix, Iy, It)
    return u, v


def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    """
    Отслеживает точку между кадрами с использованием пирамидального LK.
    """
    px, py = point
    num_levels = len(prev_pyramid)
    half_window = window_size // 2

    total_flow_x = 0.0
    total_flow_y = 0.0

    for level in range(num_levels - 1, -1, -1):
        scale_factor = (0.5 ** level)
        prev_img_level = prev_pyramid[level]
        curr_img_level = curr_pyramid[level]

        pt_x_level = px * scale_factor
        pt_y_level = py * scale_factor

        if level < num_levels - 1:
            total_flow_x *= 2.0
            total_flow_y *= 2.0

        level_flow_x = 0.0
        level_flow_y = 0.0

        for _ in range(max_iterations):
            x_prev = int(round(pt_x_level + total_flow_x)) # Используем предсказанное смещение
            y_prev = int(round(pt_y_level + total_flow_y))


            x_curr_guess = x_prev # Начальное предположение для curr_patch
            y_curr_guess = y_prev



            if not (half_window <= x_prev < prev_img_level.shape[1] - half_window and \
                    half_window <= y_prev < prev_img_level.shape[0] - half_window):
                return None

            prev_patch = prev_img_level[y_prev - half_window : y_prev + half_window + 1,
                                        x_prev - half_window : x_prev + half_window + 1]

            x_curr_iter = int(round(pt_x_level + total_flow_x + level_flow_x))
            y_curr_iter = int(round(pt_y_level + total_flow_y + level_flow_y))

            if not (half_window <= x_curr_iter < curr_img_level.shape[1] - half_window and \
                    half_window <= y_curr_iter < curr_img_level.shape[0] - half_window):
                return None # Точка или окно вышли за пределы на текущем кадре

            curr_patch = curr_img_level[y_curr_iter - half_window : y_curr_iter + half_window + 1,
                                        x_curr_iter - half_window : x_curr_iter + half_window + 1]


            if prev_patch.shape != (window_size, window_size) or curr_patch.shape != (window_size, window_size):
                return None

            du, dv = compute_lk_optical_flow_for_patch(prev_patch, curr_patch)

            if du is None or dv is None:
                return None

            level_flow_x += du
            level_flow_y += dv

            if abs(du) < epsilon and abs(dv) < epsilon:
                break
        
        total_flow_x += level_flow_x
        total_flow_y += level_flow_y


    # Финальная позиция точки на исходном изображении
    new_x = px + total_flow_x
    new_y = py + total_flow_y

    return new_x, new_y


def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    """
    Вычисляет разреженный оптический поток методом Лукаса-Канаде.
    """
    if prev_frame.ndim == 3:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    else:
        prev_gray = prev_frame.copy()
    if curr_frame.ndim == 3:
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    else:
        curr_gray = curr_frame.copy()


    prev_gray = prev_gray.astype(np.float32)
    curr_gray = curr_gray.astype(np.float32)


    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels)

    new_points_list = []
    status_list = []

    for point in points:
        px, py = point.ravel() # Убедимся, что это скаляры
        
        
        new_pt_coords = track_point_with_pyramid_lk(
            prev_pyramid, curr_pyramid, (px, py),
            window_size=window_size, max_iterations=max_iterations, epsilon=epsilon
        )

        if new_pt_coords is not None:
            new_points_list.append([new_pt_coords[0], new_pt_coords[1]])
            status_list.append(1)
        else:
            new_points_list.append([px, py]) # Возвращаем исходную точку
            status_list.append(0)

    new_points_arr = np.array(new_points_list, dtype=np.float32)
    status_arr = np.array(status_list, dtype=np.uint8)
    
    return new_points_arr, status_arr


def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Ошибка: не удалось открыть видео {video_path}")
        return None

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Используем mp4v для .mp4
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    ret, old_frame = cap.read()
    if not ret:
        print("Не удалось прочитать первый кадр")
        cap.release()
        out.release()
        return None
        
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    
    if p0 is None:
        print("Не найдено хороших точек на первом кадре.")
        cap.release()
        out.release()
        return None
    

    mask = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (len(p0), 3))

    for i in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p0_for_my_lk = p0.reshape(-1, 2) if p0.ndim == 3 else p0

        p1, st = lucas_kanade_optical_flow(
            old_gray, 
            frame_gray,
            p0_for_my_lk,
            window_size=15,
            num_pyramid_levels=3,
            max_iterations=10,
            epsilon=0.03 
        )
        

        st = st.ravel()
        
        if p1 is not None and len(p1) > 0:
            good_new = p1[st == 1]
            good_old = p0_for_my_lk[st == 1] 
            for j, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()

                current_color_index = np.where(np.all(p0_for_my_lk == old, axis=1))[0]
                if len(current_color_index) > 0:
                    color_idx = current_color_index[0] % len(color) # На случай если p0 меняется
                else:
                    color_idx = j % len(color)

                mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[color_idx].tolist(), 2)
                frame = cv2.circle(frame, (int(a), int(b)), 5, color[color_idx].tolist(), -1)

        img = cv2.add(frame, mask)
        out.write(img)
        old_gray = frame_gray.copy()
        

        if p1 is not None and len(p1) > 0:
            p0 = good_new.reshape(-1, 1, 2)
        else:
            p0 = None # Нет точек для отслеживания

        if p0 is None or len(p0) < 5:
            p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
            if p0 is None:
                print("Не удалось найти новые точки, завершение.")
                break
            mask = np.zeros_like(old_frame)
            color = np.random.randint(0, 255, (len(p0), 3))


    cap.release()
    out.release()
    print(f"Результат сохранен в {output_path}")
    return output_path

In [14]:
result_path = demo_optical_flow(video_path='/kaggle/working/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

100%|██████████| 913/913 [00:29<00:00, 30.55it/s]

Результат сохранен в output_my_LK.mp4


### Релизация OpenCV - cv2.calcOpticalFlowPyrLK

In [15]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео с использованием cv2.calcOpticalFlowPyrLK.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Параметры для Lucas-Kanade оптического потока
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    mask = np.zeros_like(old_frame)

    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Выбираем хорошие точки
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        img = cv2.add(frame, mask)

        out.write(img)

        old_gray = frame_gray.copy()

        p0 = good_new.reshape(-1, 1, 2)

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [16]:
result_path = demo_optical_flow_opencv(video_path='/kaggle/working/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:04<00:00, 224.74it/s]

Результат сохранен в output_opencv_LK.mp4


### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.

In [18]:
def demo_optical_flow_with_redetection(video_path='/kaggle/working/slow_traffic_small.mp4', 
                                        output_path='output_LK_redetection.mp4',
                                        use_my_lk=True):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Ошибка: не удалось открыть видео {video_path}")
        return None

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )
    
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    ret, old_frame = cap.read()
    if not ret:
        print("Не удалось прочитать первый кадр")
        cap.release()
        out.release()
        return None
        
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    if p0 is None:
        print("Не найдено хороших точек на первом кадре.")
        cap.release()
        out.release()
        return None

    mask = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (feature_params['maxCorners'], 3)) 

    tracked_points_info = {}
    point_id_counter = 0

    for i, pt_coords in enumerate(p0.reshape(-1,2)):
        tracked_points_info[point_id_counter] = (pt_coords, i % len(color), 1)
        point_id_counter += 1


    min_points_to_track = int(feature_params['maxCorners'] * 0.7)
    redetection_interval = 10

    for frame_idx in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        p0_current_list = [info[0] for info in tracked_points_info.values()]
        
        if not p0_current_list:
            p0_new_features = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
            if p0_new_features is not None:
                for pt_coords in p0_new_features.reshape(-1,2):
                    tracked_points_info[point_id_counter] = (pt_coords, point_id_counter % len(color), 1)
                    point_id_counter += 1
                p0_current_list = [info[0] for info in tracked_points_info.values()]
            else: 
                img = cv2.add(frame, mask)
                out.write(img)
                old_gray = frame_gray.copy()
                continue
        if not p0_current_list:
             img = cv2.add(frame, mask)
             out.write(img)
             old_gray = frame_gray.copy()
             continue

        p0_current_np = np.array(p0_current_list, dtype=np.float32).reshape(-1, 1, 2)
        
        if use_my_lk:
            p1, st_flat = lucas_kanade_optical_flow(
                old_gray, frame_gray, p0_current_np.reshape(-1,2),
                window_size=lk_params['winSize'][0], 
                num_pyramid_levels=lk_params['maxLevel'],
                max_iterations=lk_params['criteria'][1],
                epsilon=lk_params['criteria'][2]
            )
            st = st_flat.reshape(-1,1)
        else:
            p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0_current_np, None, **lk_params)

        new_tracked_points_info = {}
        if p1 is not None:
            current_point_ids = list(tracked_points_info.keys())

            for i in range(len(st)):
                if st[i] == 1:
                    point_id = current_point_ids[i]
                    old_coords, color_idx, age = tracked_points_info[point_id]
                    new_coords_lk = p1[i].ravel()

                    a, b = new_coords_lk
                    c, d = old_coords
                    
                    mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[color_idx].tolist(), 2)
                    frame = cv2.circle(frame, (int(a), int(b)), 5, color[color_idx].tolist(), -1)
                    
                    new_tracked_points_info[point_id] = (new_coords_lk, color_idx, age + 1)
        
        tracked_points_info = new_tracked_points_info # Обновляем словарь отслеживаемых точек

        if frame_idx % redetection_interval == 0 or len(tracked_points_info) < min_points_to_track:
            detection_mask = np.ones_like(old_gray)
            for pt_id in tracked_points_info:
                pt_x, pt_y = tracked_points_info[pt_id][0].astype(int)
                cv2.circle(detection_mask, (pt_x, pt_y), feature_params['minDistance'] * 2, 0, -1)

            p0_new_features = cv2.goodFeaturesToTrack(old_gray, 
                                                       mask=detection_mask, 
                                                       maxCorners=feature_params['maxCorners'] - len(tracked_points_info), # Детектируем недостающие
                                                       **{k:v for k,v in feature_params.items() if k!= 'maxCorners'}) # Передаем остальные параметры

            if p0_new_features is not None:
                for pt_coords in p0_new_features.reshape(-1,2):
                    if len(tracked_points_info) < feature_params['maxCorners']:
                        tracked_points_info[point_id_counter] = (pt_coords, point_id_counter % len(color), 1)
                        point_id_counter += 1
        
        img = cv2.add(frame, mask)
        out.write(img)
        old_gray = frame_gray.copy()

    cap.release()
    out.release()
    print(f"Результат с переобнаружением сохранен в {output_path}")
    return output_path


result_path_redetect_cv_lk = demo_optical_flow_with_redetection(use_my_lk=False, output_path='output_opencv_LK_redetection.mp4')

100%|██████████| 913/913 [00:06<00:00, 132.81it/s]

Результат с переобнаружением сохранен в output_opencv_LK_redetection.mp4


### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:**

Основное отличие между разреженным (sparse) оптическим потоком, таким как метод Лукаса-Канаде, и плотным (dense) оптическим потоком, таким как метод Фарнебак, заключается в количестве и распределении пикселей, для которых вычисляется вектор движения


## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.

# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**

Полиномиальная аппроксимация окрестностей

Наблюдение за смещением полинома

Решение для вектора смещения

Пирамидальный подход

### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:**
В методе Фарнебак, большие смещения объектов между кадрами обрабатываются с помощью пирамидального подхода

In [20]:
def demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_Farneback.mp4'):
    """
    Демонстрация работы алгоритма плотного оптического потока Farneback на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Берем первый кадр и преобразуем его в оттенки серого
    ret, frame1 = cap.read()
    if not ret:
        print('Не удалось прочитать видео')
        return None

    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    # Создаем HSV-изображение для визуализации потока
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255  # Насыщенность устанавливаем на максимум

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame2 = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток методом Farneback
        # Параметры:
        # - 0.5: коэффициент масштабирования для пирамиды изображений
        # - 3: кол-во уровней пирамиды
        # - 15: размер окна для усреднения
        # - 3: число итераций на каждом уровне пирамиды
        # - 5: размер окна для полиномиальной аппроксимации
        # - 1.2: стандартное отклонение для сглаживания
        flow = cv2.calcOpticalFlowFarneback(
            prvs, next_frame, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Преобразуем векторы потока из декартовых координат в полярные
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Кодируем направление потока как оттенок (hue)
        hsv[..., 0] = ang * 180 / np.pi / 2

        # Кодируем величину потока как яркость (value)
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        # Преобразуем HSV в BGR для отображения
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Записываем результат
        out.write(bgr)

        # Обновляем предыдущий кадр
        prvs = next_frame

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [23]:
result_path = demo_optical_flow_farneback_opencv(video_path='/kaggle/working/slow_traffic_small.mp4', output_path='output_opencv_farneback.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
  0%|          | 3/913 [00:00<01:52,  8.09it/s]


KeyboardInterrupt: 

### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:**
Предварительная обработка изображений может существенно повлиять на качество оптического потока, получаемого методом Farneback, поскольку он чувствителен к градиентам яркости и локальным структурам.